# OU generator alignment

**Paper:** *Gaeta–Lie Neural SDEs: Symmetry-Regularized Learning of Stochastic Dynamics* (NeurIPS 2026) — Section 5.7, Appendix K, Figure 10.

Discovers generators on a 1D Ornstein–Uhlenbeck SDE with a known symmetry algebra and compares the learned vector fields with their best-fit combinations of the analytic OU generators.

**Pipeline naming used in the code:** Stage I = baseline neural SDE surrogate; Stage II = symmetry discovery over the basis library; Stage F = Stage III (symmetry-regularized refinement).

Run the cells top to bottom. All configuration lives in `CONFIG` in the first code cell. See the [repository README](../../README.md) for installation and the full experiment index.

## Cell 1 — Config, imports, global knobs

In [ ]:
# ============================================================
# Cell 1 — Config, imports, global knobs
# ============================================================

from jax import config as jax_config
jax_config.update("jax_enable_x64", False)

import math
import time
from functools import partial

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

print("JAX version:", jax.__version__)
print("JAX devices:", jax.devices())


# ----------------------------
# Main configuration knobs
# ----------------------------
CONFIG = {
    # Reproducibility
    "SEED": 42,

    # Ground-truth OU SDE:
    #     dX_t = OU_MU * X_t dt + OU_SIGMA dW_t
    #
    # For Gaeta Example 7 mean-reverting OU, use OU_MU = -1.0.
    "OU_MU": -1.0,
    "OU_SIGMA": 0.7,

    # Synthetic data generation
    "T_FINAL": 2.0,
    "DT": 0.01,
    "N_TRAJECTORIES": 96,
    "X0_MEAN": 0.0,
    "X0_STD": 0.7,
    "TRAIN_FRACTION": 0.8,

    # Neural SDE surrogate
    "MLP_HIDDEN_DIM": 64,
    "MLP_DEPTH": 3,
    "INPUT_T_SCALE": 2.0,
    "INPUT_X_SCALE": 2.5,
    "SIGMA_EPS": 1e-4,

    # Baseline surrogate training
    "BASELINE_EPOCHS": 1500,
    "BASELINE_BATCH_SIZE": 512,
    "BASELINE_LR": 2e-3,
    "PRINT_EVERY": 250,

    # Symmetry discovery
    "N_GENERATORS": 3,
    "SYM_DISCOVERY_EPOCHS": 2000,
    "SYM_DISCOVERY_LR": 1e-2,
    "SYM_DISCOVERY_POINTS": 512,
    "SYM_DISCOVERY_T_BOX": (0.0, 2.0),
    "SYM_DISCOVERY_X_BOX": (-2.5, 2.5),
    "SYM_INIT_SCALE": 0.05,

    # Symmetry-discovery loss weights
    "LAMBDA_DET": 1.0,
    "LAMBDA_CLOSE": 0.1,
    "LAMBDA_IND": 0.05,
    "LAMBDA_PROJECTABLE": 1.0,
    "LAMBDA_NORM_HINGE": 0.01,
    "MIN_GENERATOR_NORM": 0.15,
    "CLOSURE_RCOND": 1e-5,

    # Include these extra Gaeta Example 8 linear-drift basis elements.
    # OU_MU is automatically included below, even if not listed here.
    "BASIS_LINEAR_MU_LIST": [-2.0, -0.5, 0.5, 2.0],

    # Exponential clipping for numerical safety in basis functions
    "BASIS_EXP_CLIP": 8.0,

    # Diagnostics and plotting
    "N_PLOT_GRID": 200,

    # User-changeable tx-grid for learned-vs-ground-truth symmetry plots
    "SYM_PLOT_N_POINTS": 150,
    "SYM_PLOT_T_BOX": (0.0, 2.0),
    "SYM_PLOT_X_BOX": (-2.5, 2.5),
    "SYM_PLOT_QUIVER_SCALE": 30,

    # Save plots to the current working directory
    "SAVE_SYM_PLOTS_DIR": ".",
}

key = jax.random.PRNGKey(CONFIG["SEED"])

## Cell 2 — Synthetic OU SDE data with known Gaeta symmetries

In [ ]:
# ============================================================
# Cell 2 — Synthetic OU SDE data with known Gaeta symmetries
# ============================================================

def true_drift(t, x):
    """
    Ground-truth OU drift:
        dX_t = OU_MU * X_t dt + OU_SIGMA dW_t

    Gaeta Example 7 mean-reverting OU uses OU_MU = -1.
    """
    return CONFIG["OU_MU"] * x


def true_sigma(t, x):
    """
    Constant OU diffusion.
    """
    return CONFIG["OU_SIGMA"] + 0.0 * x


@jax.jit
def simulate_sde_paths(key):
    """
    Simulate all trajectories using Euler-Maruyama.

    Returns
    -------
    times : shape (n_steps + 1,)
    paths : shape (n_paths, n_steps + 1)
    """
    T = CONFIG["T_FINAL"]
    dt = CONFIG["DT"]
    n_steps = int(round(T / dt))
    n_paths = CONFIG["N_TRAJECTORIES"]

    key_x0, key_noise = jax.random.split(key)
    x0 = CONFIG["X0_MEAN"] + CONFIG["X0_STD"] * jax.random.normal(key_x0, (n_paths,))
    eps = jax.random.normal(key_noise, (n_steps, n_paths))

    times = jnp.arange(n_steps + 1, dtype=jnp.float32) * dt
    sqrt_dt = jnp.sqrt(dt)

    def step(x, inp):
        t, noise = inp
        f = true_drift(t, x)
        s = true_sigma(t, x)
        x_next = x + f * dt + s * sqrt_dt * noise
        return x_next, x_next

    _, xs_tail = jax.lax.scan(step, x0, (times[:-1], eps))
    paths = jnp.concatenate([x0[None, :], xs_tail], axis=0).T
    return times, paths


def make_increment_dataset(times, paths):
    """
    Convert simulated trajectories into one-step increment data.

    Returns tuple:
        t0, x0, dt, dx
    each shape (n_samples,)
    """
    n_paths, n_times = paths.shape
    n_steps = n_times - 1
    dt = CONFIG["DT"]

    t0 = jnp.broadcast_to(times[:-1][None, :], (n_paths, n_steps)).reshape(-1)
    x0 = paths[:, :-1].reshape(-1)
    dx = (paths[:, 1:] - paths[:, :-1]).reshape(-1)
    dt_arr = jnp.full_like(dx, dt)

    return t0, x0, dt_arr, dx


def train_test_split_dataset(key, dataset, train_fraction):
    n = dataset[0].shape[0]
    perm = jax.random.permutation(key, n)
    n_train = int(train_fraction * n)

    idx_train = perm[:n_train]
    idx_test = perm[n_train:]

    train = tuple(arr[idx_train] for arr in dataset)
    test = tuple(arr[idx_test] for arr in dataset)
    return train, test


def sample_batch(key, dataset, batch_size):
    n = dataset[0].shape[0]
    idx = jax.random.randint(key, (batch_size,), 0, n)
    return tuple(arr[idx] for arr in dataset)


key, subkey = jax.random.split(key)
times, paths = simulate_sde_paths(subkey)
dataset = make_increment_dataset(times, paths)

key, subkey = jax.random.split(key)
train_data, test_data = train_test_split_dataset(
    subkey, dataset, CONFIG["TRAIN_FRACTION"]
)

print("OU_MU:", CONFIG["OU_MU"])
print("OU_SIGMA:", CONFIG["OU_SIGMA"])
print("times shape:", times.shape)
print("paths shape:", paths.shape)
print("train samples:", train_data[0].shape[0])
print("test samples:", test_data[0].shape[0])

plt.figure(figsize=(8, 4))
for i in range(min(12, CONFIG["N_TRAJECTORIES"])):
    plt.plot(np.array(times), np.array(paths[i]), alpha=0.6)
plt.xlabel("t")
plt.ylabel("X_t")
plt.title("Synthetic OU SDE trajectories")
plt.grid(True)
plt.show()

## Cell 3 — Neural SDE surrogate, Adam optimizer, baseline training

In [ ]:
# ============================================================
# Cell 3 — Neural SDE surrogate, Adam optimizer, baseline training
# ============================================================

def glorot_uniform(key, in_dim, out_dim):
    lim = jnp.sqrt(6.0 / (in_dim + out_dim))
    return jax.random.uniform(key, (in_dim, out_dim), minval=-lim, maxval=lim)


class MLP:
    """
    Small pure-JAX MLP class.

    Output dimension is 2:
        raw[0] -> drift f_theta(t,x)
        raw[1] -> raw diffusion, transformed by softplus
    """

    def __init__(self, in_dim=2, hidden_dim=64, depth=3, out_dim=2):
        self.in_dim = in_dim
        self.hidden_dim = hidden_dim
        self.depth = depth
        self.out_dim = out_dim

    def init(self, key):
        sizes = [self.in_dim] + [self.hidden_dim] * self.depth + [self.out_dim]
        keys = jax.random.split(key, len(sizes) - 1)

        params = []
        for k, in_d, out_d in zip(keys, sizes[:-1], sizes[1:]):
            W = glorot_uniform(k, in_d, out_d)
            b = jnp.zeros((out_d,), dtype=jnp.float32)
            params.append({"W": W, "b": b})
        return params

    def apply(self, params, z):
        h = z
        for layer in params[:-1]:
            h = jnp.tanh(h @ layer["W"] + layer["b"])
        out = h @ params[-1]["W"] + params[-1]["b"]
        return out


surrogate_model = MLP(
    in_dim=2,
    hidden_dim=CONFIG["MLP_HIDDEN_DIM"],
    depth=CONFIG["MLP_DEPTH"],
    out_dim=2,
)


def normalize_input(t, x):
    tn = t / CONFIG["INPUT_T_SCALE"]
    xn = x / CONFIG["INPUT_X_SCALE"]
    return jnp.array([tn, xn], dtype=jnp.float32)


def surrogate_f_sigma(params, t, x):
    z = normalize_input(t, x)
    raw = surrogate_model.apply(params, z)

    f = raw[0]
    sigma = jax.nn.softplus(raw[1]) + CONFIG["SIGMA_EPS"]
    return f, sigma


def f_single(params, z):
    return surrogate_f_sigma(params, z[0], z[1])[0]


def sigma_single(params, z):
    return surrogate_f_sigma(params, z[0], z[1])[1]


def data_nll_loss(params, batch):
    """
    Euler-Maruyama Gaussian negative log-likelihood.
    batch = (t0, x0, dt, dx)
    """
    t0, x0, dt, dx = batch

    f, sigma = jax.vmap(lambda tt, xx: surrogate_f_sigma(params, tt, xx))(t0, x0)
    mean = f * dt
    var = sigma**2 * dt + 1e-8

    nll = 0.5 * ((dx - mean) ** 2 / var + jnp.log(2.0 * jnp.pi * var))
    return jnp.mean(nll)


# ----------------------------
# Lightweight pure-JAX Adam
# ----------------------------
def adam_init(params):
    m = jax.tree_util.tree_map(jnp.zeros_like, params)
    v = jax.tree_util.tree_map(jnp.zeros_like, params)
    t = jnp.array(0, dtype=jnp.int32)
    return {"m": m, "v": v, "t": t}


def adam_update(params, grads, state, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    t_new = state["t"] + 1

    m_new = jax.tree_util.tree_map(
        lambda m, g: beta1 * m + (1.0 - beta1) * g,
        state["m"],
        grads,
    )
    v_new = jax.tree_util.tree_map(
        lambda v, g: beta2 * v + (1.0 - beta2) * (g * g),
        state["v"],
        grads,
    )

    m_hat = jax.tree_util.tree_map(lambda m: m / (1.0 - beta1**t_new), m_new)
    v_hat = jax.tree_util.tree_map(lambda v: v / (1.0 - beta2**t_new), v_new)

    params_new = jax.tree_util.tree_map(
        lambda p, m, v: p - lr * m / (jnp.sqrt(v) + eps),
        params,
        m_hat,
        v_hat,
    )

    return params_new, {"m": m_new, "v": v_new, "t": t_new}


@jax.jit
def baseline_train_step(params, opt_state, batch):
    loss, grads = jax.value_and_grad(data_nll_loss)(params, batch)
    params, opt_state = adam_update(
        params, grads, opt_state, CONFIG["BASELINE_LR"]
    )
    return params, opt_state, loss


# ----------------------------
# Train baseline surrogate
# ----------------------------
key, subkey = jax.random.split(key)
sde_params = surrogate_model.init(subkey)
sde_opt_state = adam_init(sde_params)

baseline_losses = []
t_start = time.time()

for epoch in range(1, CONFIG["BASELINE_EPOCHS"] + 1):
    key, subkey = jax.random.split(key)
    batch = sample_batch(subkey, train_data, CONFIG["BASELINE_BATCH_SIZE"])

    sde_params, sde_opt_state, loss = baseline_train_step(
        sde_params, sde_opt_state, batch
    )

    baseline_losses.append(float(loss))

    if epoch % CONFIG["PRINT_EVERY"] == 0 or epoch == 1:
        train_loss = float(data_nll_loss(sde_params, train_data))
        test_loss = float(data_nll_loss(sde_params, test_data))
        print(
            f"[Baseline] epoch {epoch:5d} | "
            f"batch nll {float(loss): .5f} | "
            f"train nll {train_loss: .5f} | "
            f"test nll {test_loss: .5f}"
        )

print(f"Baseline training time: {time.time() - t_start:.2f} sec")

## Cell 4 — Editable Gaeta SDE symmetry-basis library

In [ ]:
# ============================================================
# Cell 4 — Editable Gaeta SDE symmetry-basis library
# ============================================================

def safe_exp(z):
    c = CONFIG["BASIS_EXP_CLIP"]
    return jnp.exp(jnp.clip(z, -c, c))


# ------------------------------------------------------------------
# Editable symmetry basis library
#
# Each basis function returns a 2-vector:
#     [tau(t), xi(t,x)]
#
# The learned generator is:
#     X_a(t,x) = sum_r W[a,r] B_r(t,x)
#
# W[a,r] are trainable scalar constants, not functions of (t,x).
#
# The library below includes the 1D-compatible SDE-level generators
# derived or implied by Gaeta's examples.
#
# 2D examples such as Kramers/Finkel-type systems are not directly
# inserted into this 1D notebook, because their generators act on
# more than one spatial coordinate.
# ------------------------------------------------------------------

BASIS_LIBRARY = []


def add_basis(name, source, fn):
    BASIS_LIBRARY.append({"name": name, "source": source, "fn": fn})


# ----------------------------
# Example 1: Brownian motion dx = sigma0 dW
# SDE symmetries: ∂t, ∂x, 2t∂t + x∂x
# ----------------------------
add_basis(
    "E1_time_translation__dt",
    "Gaeta Example 1: Brownian SDE",
    lambda t, x: jnp.array([1.0, 0.0], dtype=jnp.float32),
)

add_basis(
    "E1_space_translation__dx",
    "Gaeta Example 1: Brownian SDE",
    lambda t, x: jnp.array([0.0, 1.0], dtype=jnp.float32),
)

add_basis(
    "E1_brownian_scaling__2t_dt_plus_x_dx",
    "Gaeta Example 1: Brownian SDE",
    lambda t, x: jnp.array([2.0 * t, x], dtype=jnp.float32),
)


# ----------------------------
# Example 2: dx = dt + x dW
# SDE symmetry is only time translation ∂t.
# Already included above, but retained conceptually in comments.
# ----------------------------


# ----------------------------
# Example 3: dx = x dt + dW
# SDE symmetries:
#   ∂t,
#   e^t ∂x,
#   e^{2t}(∂t + x∂x)
# ----------------------------
add_basis(
    "E3_exp_t_space__e_t_dx",
    "Gaeta Example 3: dx = x dt + dW",
    lambda t, x: jnp.array([0.0, safe_exp(t)], dtype=jnp.float32),
)

add_basis(
    "E3_exp_2t_projective__e2t_dt_plus_e2t_x_dx",
    "Gaeta Example 3: dx = x dt + dW",
    lambda t, x: jnp.array([safe_exp(2.0 * t), safe_exp(2.0 * t) * x], dtype=jnp.float32),
)


# ----------------------------
# Example 7: 1D version of n uncoupled Langevin oscillators
# dx = -x dt + sqrt(2s) dW
# SDE symmetries:
#   ∂t,
#   e^{-t} ∂x,
#   e^{-2t}(∂t - x∂x)
# ----------------------------
add_basis(
    "E7_exp_minus_t_space__em_t_dx",
    "Gaeta Example 7: dx = -x dt + const dW",
    lambda t, x: jnp.array([0.0, safe_exp(-t)], dtype=jnp.float32),
)

add_basis(
    "E7_exp_minus_2t_projective__em2t_dt_minus_em2t_x_dx",
    "Gaeta Example 7: dx = -x dt + const dW",
    lambda t, x: jnp.array([safe_exp(-2.0 * t), -safe_exp(-2.0 * t) * x], dtype=jnp.float32),
)


# ----------------------------
# Example 8: linear drift f(x)=mu*x, constant diffusion.
# 1D-compatible family:
#   e^{mu t} ∂x,
#   e^{2mu t}(∂t + mu x∂x)
#
# The explicit mu=+1 and mu=-1 cases above correspond to Examples 3 and 7.
# Here we add extra user-configurable mu values to enlarge the editable dictionary.
# ----------------------------
def make_linear_mu_space(mu):
    return lambda t, x: jnp.array([0.0, safe_exp(mu * t)], dtype=jnp.float32)


def make_linear_mu_projective(mu):
    return lambda t, x: jnp.array(
        [safe_exp(2.0 * mu * t), mu * safe_exp(2.0 * mu * t) * x],
        dtype=jnp.float32,
    )


# Always include the user-selected OU_MU in the linear-drift dictionary.
linear_mu_values = list(CONFIG["BASIS_LINEAR_MU_LIST"])
if abs(CONFIG["OU_MU"]) > 1e-8:
    linear_mu_values.append(float(CONFIG["OU_MU"]))

# Deduplicate while preserving order.
seen_mu = set()
linear_mu_values_unique = []
for mu in linear_mu_values:
    key_mu = round(float(mu), 8)
    if key_mu not in seen_mu:
        seen_mu.add(key_mu)
        linear_mu_values_unique.append(float(mu))


for mu in linear_mu_values_unique:
    if abs(mu) > 1e-8:
        add_basis(
            f"E8_linear_mu_{mu:+.2f}__e_mut_dx",
            "Gaeta Example 8: linear drift constant diffusion, 1D family",
            make_linear_mu_space(float(mu)),
        )
        add_basis(
            f"E8_linear_mu_{mu:+.2f}__e_2mut_dt_plus_mu_x_dx",
            "Gaeta Example 8: linear drift constant diffusion, 1D family",
            make_linear_mu_projective(float(mu)),
        )


# ----------------------------
# Example 10: nonlinear radial system.
# In 1D with lambda != 0, only ∂t survives.
# Already included as E1_time_translation__dt.
# ----------------------------


BASIS_NAMES = [b["name"] for b in BASIS_LIBRARY]
BASIS_SOURCES = [b["source"] for b in BASIS_LIBRARY]
BASIS_FNS = tuple(b["fn"] for b in BASIS_LIBRARY)
N_BASIS = len(BASIS_FNS)

print(f"Number of basis functions: {N_BASIS}")
for i, (nm, src) in enumerate(zip(BASIS_NAMES, BASIS_SOURCES)):
    print(f"{i:02d} | {nm} | {src}")


def eval_basis_one(t, x):
    """
    Returns basis values at one point.

    Output shape:
        (R, 2)
    where component 0 is tau and component 1 is xi.
    """
    return jnp.stack([fn(t, x) for fn in BASIS_FNS], axis=0)


def eval_generators_point(W, z):
    """
    Evaluate learned generators at one point.

    W shape:
        (m, R)

    z shape:
        (2,) = [t, x]

    Returns:
        X shape (m, 2), with X[a] = [tau_a, xi_a].
    """
    t, x = z[0], z[1]
    B = eval_basis_one(t, x)       # (R, 2)
    X = W @ B                      # (m, 2)
    return X


def eval_generators_batch(W, tx):
    """
    tx shape:
        (S, 2)

    Returns:
        X shape (S, m, 2)
    """
    return jax.vmap(lambda z: eval_generators_point(W, z))(tx)


def sample_tx_box(key, n_points, t_box, x_box):
    key_t, key_x = jax.random.split(key)
    t = jax.random.uniform(
        key_t,
        (n_points,),
        minval=float(t_box[0]),
        maxval=float(t_box[1]),
    )
    x = jax.random.uniform(
        key_x,
        (n_points,),
        minval=float(x_box[0]),
        maxval=float(x_box[1]),
    )
    return jnp.stack([t, x], axis=1)


key, subkey = jax.random.split(key)
W = CONFIG["SYM_INIT_SCALE"] * jax.random.normal(
    subkey,
    (CONFIG["N_GENERATORS"], N_BASIS),
)

print("Initial W shape:", W.shape)

## Cell 5 — Gaeta determining loss, global Lie closure, symmetry discovery

In [ ]:
# ============================================================
# Cell 5 — Gaeta determining loss, global Lie closure, symmetry discovery
# ============================================================

def det_residuals_point(sde_params, W, z):
    """
    1D specialization of Gaeta's projectable Ito SDE determining equations.

    Candidate generator:
        X_a = tau_a(t) ∂t + xi_a(t,x) ∂x

    Neural SDE:
        dX_t = f_theta(t,X_t) dt + sigma_theta(t,X_t) dW_t

    Residuals:
        R_f =
            xi_t + f xi_x - xi f_x - ∂t(f tau)
            + 0.5 sigma^2 xi_xx

        R_sigma =
            sigma xi_x - xi sigma_x - tau sigma_t
            - 0.5 sigma tau_t

    Returns:
        R_f shape (m,)
        R_sigma shape (m,)
        tau_x shape (m,)
    """
    f = f_single(sde_params, z)
    sigma = sigma_single(sde_params, z)

    grad_f = jax.grad(lambda zz: f_single(sde_params, zz))(z)
    grad_sigma = jax.grad(lambda zz: sigma_single(sde_params, zz))(z)

    f_t, f_x = grad_f[0], grad_f[1]
    sigma_t, sigma_x = grad_sigma[0], grad_sigma[1]

    gen_fun = lambda zz: eval_generators_point(W, zz)
    X = gen_fun(z)                              # (m, 2)
    J = jax.jacfwd(gen_fun)(z)                  # (m, 2, 2)
    H = jax.jacfwd(jax.jacfwd(gen_fun))(z)      # (m, 2, 2, 2)

    tau = X[:, 0]
    xi = X[:, 1]

    tau_t = J[:, 0, 0]
    tau_x = J[:, 0, 1]
    xi_t = J[:, 1, 0]
    xi_x = J[:, 1, 1]
    xi_xx = H[:, 1, 1, 1]

    R_f = (
        xi_t
        + f * xi_x
        - xi * f_x
        - (f_t * tau + f * tau_t)
        + 0.5 * sigma**2 * xi_xx
    )

    R_sigma = (
        sigma * xi_x
        - xi * sigma_x
        - tau * sigma_t
        - 0.5 * sigma * tau_t
    )

    return R_f, R_sigma, tau_x


def determining_loss(sde_params, W, tx):
    R_f, R_sigma, tau_x = jax.vmap(
        lambda z: det_residuals_point(sde_params, W, z)
    )(tx)

    det = jnp.mean(R_f**2 + R_sigma**2)
    projectable = jnp.mean(tau_x**2)
    return det, projectable


def generator_jacobians_batch(W, tx):
    """
    Returns:
        X shape (S, m, 2)
        J shape (S, m, 2, 2)
    """
    def one(z):
        gen_fun = lambda zz: eval_generators_point(W, zz)
        X = gen_fun(z)
        J = jax.jacfwd(gen_fun)(z)
        return X, J

    return jax.vmap(one)(tx)


def lie_closure_loss_global(W, tx):
    """
    Global-batch Lie closure.

    We solve one least-squares problem over the whole collocation batch:

        [X_i, X_j] ≈ sum_k C_ij^k X_k

    This makes the structure constants global by construction.
    Therefore we do NOT need a separate structure-constant constancy loss.
    """
    m = W.shape[0]
    if m < 2:
        return jnp.array(0.0, dtype=jnp.float32)

    X, J = generator_jacobians_batch(W, tx)
    S = tx.shape[0]

    idx_i, idx_j = jnp.triu_indices(m, k=1)
    P = idx_i.shape[0]

    Xi = X[:, idx_i, :]     # (S, P, 2)
    Xj = X[:, idx_j, :]     # (S, P, 2)
    Ji = J[:, idx_i, :, :]  # (S, P, 2, 2)
    Jj = J[:, idx_j, :, :]  # (S, P, 2, 2)

    tau_i, xi_i = Xi[..., 0], Xi[..., 1]
    tau_j, xi_j = Xj[..., 0], Xj[..., 1]

    tau_i_t = Ji[..., 0, 0]
    tau_i_x = Ji[..., 0, 1]
    xi_i_t = Ji[..., 1, 0]
    xi_i_x = Ji[..., 1, 1]

    tau_j_t = Jj[..., 0, 0]
    tau_j_x = Jj[..., 0, 1]
    xi_j_t = Jj[..., 1, 0]
    xi_j_x = Jj[..., 1, 1]

    bracket_tau = (
        tau_i * tau_j_t
        + xi_i * tau_j_x
        - tau_j * tau_i_t
        - xi_j * tau_i_x
    )

    bracket_xi = (
        tau_i * xi_j_t
        + xi_i * xi_j_x
        - tau_j * xi_i_t
        - xi_j * xi_i_x
    )

    brackets = jnp.stack([bracket_tau, bracket_xi], axis=-1)  # (S, P, 2)

    # V has columns equal to flattened generator fields.
    # V shape: (2S, m)
    V = jnp.transpose(X, (1, 0, 2)).reshape(m, -1).T

    # B has columns equal to flattened brackets.
    # B shape: (2S, P)
    B = jnp.transpose(brackets, (1, 0, 2)).reshape(P, -1).T

    G = V.T @ V
    reg = (CONFIG["CLOSURE_RCOND"] ** 2) * jnp.eye(m, dtype=G.dtype)
    rhs = V.T @ B

    C = jnp.linalg.solve(G + reg, rhs)   # shape (m, P)
    B_proj = V @ C
    residual = B - B_proj

    return jnp.mean(residual**2)


def gram_independence_loss(W, tx):
    """
    Function-space Gram orthonormality:
        G_ab = E[tau_a tau_b + xi_a xi_b]
    """
    X = eval_generators_batch(W, tx)  # (S, m, 2)
    S = X.shape[0]
    m = X.shape[1]

    V = jnp.transpose(X, (1, 0, 2)).reshape(m, -1).T  # (2S, m)
    G = (V.T @ V) / (2.0 * S)

    I = jnp.eye(m, dtype=G.dtype)
    loss_ind = jnp.mean((G - I) ** 2)

    diag = jnp.diag(G)
    norm_hinge = jnp.mean(
        jnp.maximum(0.0, CONFIG["MIN_GENERATOR_NORM"] - diag) ** 2
    )

    return loss_ind, norm_hinge, G


def sym_discovery_objective(W, sde_params, tx):
    det, proj = determining_loss(sde_params, W, tx)
    close = lie_closure_loss_global(W, tx)
    ind, norm_hinge, _ = gram_independence_loss(W, tx)

    total = (
        CONFIG["LAMBDA_DET"] * det
        + CONFIG["LAMBDA_CLOSE"] * close
        + CONFIG["LAMBDA_IND"] * ind
        + CONFIG["LAMBDA_PROJECTABLE"] * proj
        + CONFIG["LAMBDA_NORM_HINGE"] * norm_hinge
    )

    aux = {
        "total": total,
        "det": det,
        "close": close,
        "ind": ind,
        "projectable": proj,
        "norm_hinge": norm_hinge,
    }
    return total, aux


sym_value_and_grad = jax.value_and_grad(
    lambda W, sde_params, tx: sym_discovery_objective(W, sde_params, tx)[0]
)


@jax.jit
def sym_train_step(W, opt_state, sde_params, tx):
    loss, grads = sym_value_and_grad(W, sde_params, tx)
    W, opt_state = adam_update(
        W,
        grads,
        opt_state,
        CONFIG["SYM_DISCOVERY_LR"],
    )
    _, aux = sym_discovery_objective(W, sde_params, tx)
    return W, opt_state, aux


# ----------------------------
# Train symmetry basis weights W
# ----------------------------
W_opt_state = adam_init(W)
sym_history = []

t_start = time.time()

for epoch in range(1, CONFIG["SYM_DISCOVERY_EPOCHS"] + 1):
    key, subkey = jax.random.split(key)
    tx_disc = sample_tx_box(
        subkey,
        CONFIG["SYM_DISCOVERY_POINTS"],
        CONFIG["SYM_DISCOVERY_T_BOX"],
        CONFIG["SYM_DISCOVERY_X_BOX"],
    )

    W, W_opt_state, aux = sym_train_step(W, W_opt_state, sde_params, tx_disc)

    if epoch % CONFIG["PRINT_EVERY"] == 0 or epoch == 1:
        aux_np = {k: float(v) for k, v in aux.items()}
        sym_history.append(aux_np)
        print(
            f"[SymDiscovery] epoch {epoch:5d} | "
            f"total {aux_np['total']: .4e} | "
            f"det {aux_np['det']: .4e} | "
            f"close {aux_np['close']: .4e} | "
            f"ind {aux_np['ind']: .4e} | "
            f"proj {aux_np['projectable']: .4e}"
        )

print(f"Symmetry discovery time: {time.time() - t_start:.2f} sec")

# Inspect learned coefficients
W_np = np.array(W)
print("\nLearned W shape:", W_np.shape)

for a in range(CONFIG["N_GENERATORS"]):
    print(f"\nGenerator X_{a+1}: top basis coefficients")
    order = np.argsort(-np.abs(W_np[a]))
    for idx in order[:min(8, N_BASIS)]:
        print(f"  {idx:02d} | coeff {W_np[a, idx]: .5f} | {BASIS_NAMES[idx]}")

# Inspect Gram matrix
key, subkey = jax.random.split(key)
tx_eval_sym = sample_tx_box(
    subkey,
    1024,
    CONFIG["SYM_DISCOVERY_T_BOX"],
    CONFIG["SYM_DISCOVERY_X_BOX"],
)
_, _, G_eval = gram_independence_loss(W, tx_eval_sym)
print("\nGenerator Gram matrix on eval collocation points:")
print(np.array(G_eval))

## Cell 7 — Learned vs ground-truth OU symmetry diagnostics and plots

In [ ]:
# ============================================================
# Cell 7 — Learned vs ground-truth OU symmetry diagnostics and plots
# ============================================================

import os

os.makedirs(CONFIG["SAVE_SYM_PLOTS_DIR"], exist_ok=True)


# ============================================================
# Local plotting knobs for this cell
# ============================================================

# Number of sampled tx-points used in each generator comparison plot.
# Each plot shows this many blue learned arrows and this many red GT-fit arrows.
N_ARROWS_PER_FIELD = 300

# Normalized arrow length for quiver plots.
# Increase for longer arrows, decrease for shorter arrows.
ARROW_LENGTH = 0.05

# Arrow width.
ARROW_WIDTH = 0.0025


# ============================================================
# Ground-truth OU SDE generators
# ============================================================

def gt_ou_generators_point(z):
    """
    Ground-truth projectable SDE symmetry generators for
        dX_t = mu X_t dt + sigma dW_t

    In 1D:
        G1 = ∂t
        G2 = exp(mu t) ∂x
        G3 = exp(2 mu t) (∂t + mu x ∂x)

    For Gaeta Example 7, mu = -1:
        G1 = ∂t
        G2 = exp(-t) ∂x
        G3 = exp(-2t) (∂t - x ∂x)

    Returns shape:
        (3, 2), where columns are [tau, xi].
    """
    mu = CONFIG["OU_MU"]
    t, x = z[0], z[1]

    return jnp.stack(
        [
            jnp.array([1.0, 0.0], dtype=jnp.float32),
            jnp.array([0.0, safe_exp(mu * t)], dtype=jnp.float32),
            jnp.array(
                [safe_exp(2.0 * mu * t), mu * x * safe_exp(2.0 * mu * t)],
                dtype=jnp.float32,
            ),
        ],
        axis=0,
    )


def gt_ou_generators_batch(tx):
    """
    tx shape:
        (S, 2)

    Returns:
        G shape (S, 3, 2)
    """
    return jax.vmap(gt_ou_generators_point)(tx)


GT_GENERATOR_NAMES = [
    "GT_1 = ∂t",
    "GT_2 = exp(mu t) ∂x",
    "GT_3 = exp(2 mu t)(∂t + mu x ∂x)",
]


# ============================================================
# Helper functions
# ============================================================

def flatten_generator_fields(X):
    """
    X shape:
        (S, m, 2)

    Returns:
        V shape:
            (2S, m)
    """
    S = X.shape[0]
    m = X.shape[1]
    return jnp.transpose(X, (1, 0, 2)).reshape(m, -1).T


def best_gt_match_to_learned(X_learned, X_gt, ridge=1e-8):
    """
    Find the best global linear combination of GT generators matching each learned generator.

        learned_j ≈ sum_k A[k, j] GT_k

    This is necessary because symmetry discovery identifies a basis for the Lie algebra,
    not a uniquely ordered/scaled generator list.

    Inputs:
        X_learned shape (S, m, 2)
        X_gt      shape (S, g, 2)

    Returns:
        X_gt_fit shape (S, m, 2)
        A        shape (g, m)
        rel_errs shape (m,)
    """
    V_learned = flatten_generator_fields(X_learned)
    V_gt = flatten_generator_fields(X_gt)

    g = V_gt.shape[1]
    A = jnp.linalg.solve(
        V_gt.T @ V_gt + ridge * jnp.eye(g, dtype=V_gt.dtype),
        V_gt.T @ V_learned,
    )

    V_gt_fit = V_gt @ A

    S = X_learned.shape[0]
    m = X_learned.shape[1]
    X_gt_fit = jnp.transpose(V_gt_fit.T.reshape(m, S, 2), (1, 0, 2))

    rel_errs = jnp.sqrt(
        jnp.sum((V_learned - V_gt_fit) ** 2, axis=0)
        / (jnp.sum(V_learned ** 2, axis=0) + 1e-12)
    )

    return X_gt_fit, A, rel_errs


def normalize_vector_field_np(X, arrow_length=0.08, eps=1e-12):
    """
    Normalize a vector field for plotting only.

    X shape:
        (S, m, 2)

    Returns:
        X_plot shape (S, m, 2), where every nonzero vector has length arrow_length.
    """
    norms = np.linalg.norm(X, axis=-1, keepdims=True)
    return arrow_length * X / (norms + eps)


# ============================================================
# Evaluate learned and ground-truth generators on same tx-grid
# ============================================================

key, subkey = jax.random.split(key)
tx_sym_plot = sample_tx_box(
    subkey,
    N_ARROWS_PER_FIELD,
    CONFIG["SYM_PLOT_T_BOX"],
    CONFIG["SYM_PLOT_X_BOX"],
)

X_learned = eval_generators_batch(W, tx_sym_plot)      # (S, m, 2)
X_gt = gt_ou_generators_batch(tx_sym_plot)              # (S, 3, 2)

X_gt_fit, gt_to_learned_A, learned_vs_gt_rel_errs = best_gt_match_to_learned(
    X_learned,
    X_gt,
)

print("\n" + "=" * 72)
print("LEARNED GENERATOR SUBSPACE VS GROUND-TRUTH OU GENERATORS")
print("=" * 72)
print("Rows are ground-truth generators; columns are learned generators.")
print("learned_j ≈ sum_k A[k, j] GT_k")
print(np.array(gt_to_learned_A))

print("\nRelative L2 error of each learned generator after best GT-basis fit:")
for a in range(CONFIG["N_GENERATORS"]):
    print(f"  X_{a+1}: {float(learned_vs_gt_rel_errs[a]): .6e}")


# Also check the reverse direction: can learned generators span GT generators?
X_learned_fit_to_gt, learned_to_gt_A, gt_vs_learned_rel_errs = best_gt_match_to_learned(
    X_gt,
    X_learned,
)

print("\nRows are learned generators; columns are ground-truth generators.")
print("GT_j ≈ sum_k B[k, j] learned_k")
print(np.array(learned_to_gt_A))

print("\nRelative L2 error of each GT generator after best learned-basis fit:")
for a, name in enumerate(GT_GENERATOR_NAMES):
    print(f"  {name}: {float(gt_vs_learned_rel_errs[a]): .6e}")


# ============================================================
# Plot symmetry discovery losses only
# ============================================================

if len(sym_history) > 0:
    plt.figure(figsize=(8, 4))
    plt.plot([h["det"] for h in sym_history], label="determining")
    plt.plot([h["close"] for h in sym_history], label="closure")
    plt.plot([h["ind"] for h in sym_history], label="independence")
    plt.yscale("log")
    plt.xlabel(f"checkpoint every {CONFIG['PRINT_EVERY']} epochs")
    plt.ylabel("loss")
    plt.title("Symmetry discovery losses")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    out_path = os.path.join(CONFIG["SAVE_SYM_PLOTS_DIR"], "ou_symmetry_discovery_losses.png")
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    print(f"Saved: {out_path}")
    plt.show()


# ============================================================
# Plot learned generators against best-fit ground-truth generators
#
# Arrows are normalized for plotting only:
#   Blue = learned arrows
#   Red  = best-fit ground-truth arrows
# ============================================================

tx_np = np.array(tx_sym_plot)
X_learned_np = np.array(X_learned)
X_gt_fit_np = np.array(X_gt_fit)

X_learned_plot_np = normalize_vector_field_np(
    X_learned_np,
    arrow_length=ARROW_LENGTH,
)

X_gt_fit_plot_np = normalize_vector_field_np(
    X_gt_fit_np,
    arrow_length=ARROW_LENGTH,
)

for a in range(CONFIG["N_GENERATORS"]):
    plt.figure(figsize=(7, 5))

    plt.quiver(
        tx_np[:, 0],
        tx_np[:, 1],
        X_learned_plot_np[:, a, 0],
        X_learned_plot_np[:, a, 1],
        angles="xy",
        scale_units="xy",
        scale=1.0,
        width=ARROW_WIDTH,
        alpha=0.55,
        color="blue",
        label=f"Learned X_{a+1}",
    )

    plt.quiver(
        tx_np[:, 0],
        tx_np[:, 1],
        X_gt_fit_plot_np[:, a, 0],
        X_gt_fit_plot_np[:, a, 1],
        angles="xy",
        scale_units="xy",
        scale=1.0,
        width=ARROW_WIDTH,
        alpha=0.55,
        color="red",
        label=f"Best GT fit to X_{a+1}",
    )

    plt.xlabel("t")
    plt.ylabel("x")
    plt.title(
        f"Learned X_{a+1} vs Best-Fit OU Ground Truth\n"
    )
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    out_path = os.path.join(
        CONFIG["SAVE_SYM_PLOTS_DIR"],
        f"Vis_3_X{a+1}.png",
    )
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    print(f"Saved: {out_path}")
    plt.show()


# ============================================================
# Compact final report
# ============================================================

W_np = np.array(W)

print("\n" + "=" * 72)
print("COMPACT FINAL REPORT: OU SYMMETRY DISCOVERY")
print("=" * 72)

print(f"OU_MU: {CONFIG['OU_MU']}")
print(f"OU_SIGMA: {CONFIG['OU_SIGMA']}")
print(f"N_GENERATORS learned: {CONFIG['N_GENERATORS']}")
print(f"N_BASIS in editable Gaeta library: {N_BASIS}")
print(f"Number of arrows per field: {N_ARROWS_PER_FIELD}")
print(f"Total arrows per comparison plot: {2 * N_ARROWS_PER_FIELD}")
print(f"Symmetry plot t-box: {CONFIG['SYM_PLOT_T_BOX']}")
print(f"Symmetry plot x-box: {CONFIG['SYM_PLOT_X_BOX']}")
print(f"Normalized arrow length: {ARROW_LENGTH}")
print(f"Saved plot directory: {CONFIG['SAVE_SYM_PLOTS_DIR']}")

print("\nTop coefficients per learned generator:")
for a in range(CONFIG["N_GENERATORS"]):
    order = np.argsort(-np.abs(W_np[a]))
    print(f"\nX_{a+1}:")
    for idx in order[:8]:
        print(f"  coeff {W_np[a, idx]: .5f} | {BASIS_NAMES[idx]}")